# RGN2 prediction with AlphaFold-based refinement

Select **Runtime → Change runtime type → GPU** in Google Colab, then run the cells in order.
Edit the sequence in step 2. The final PDB is an RGN2 prediction refined using AlphaFold.

This notebook retains the legacy package versions used by the working notebook. Downloads require internet access and several GB of storage. A new Colab runtime needs setup again. Existing model downloads are reused when valid.

For another sequence in the same runtime, run steps 2–8. Step 3 replaces the previous embeddings; prediction outputs for other sequence IDs are retained. The default RGN2 model omits two edge residues from its coordinate output.

Generated from the supplied working notebook. Code and notebook structure were checked locally; the complete GPU pipeline has not been rerun for this cleaned version.

# Step 1 — Install software and model resources

In [ ]:
%%bash
set -euo pipefail

if [ ! -x /opt/conda/bin/conda ]; then
    curl -fL --retry 3 https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -o /tmp/rgn2-miniconda.sh
    bash /tmp/rgn2-miniconda.sh -b -p /opt/conda
fi

/opt/conda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
/opt/conda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
source /opt/conda/etc/profile.d/conda.sh
cd /content

if [ ! -d /content/rgn2/.git ]; then
    git clone https://github.com/aqlaboratory/rgn2.git /content/rgn2
fi
if [ ! -x /opt/conda/envs/rgn2/bin/python ]; then
    conda env create -f /content/rgn2/environment.yml
fi
conda activate rgn2
python -m pip install 'pip<24'
python -m pip install numpy==1.18.5 gast==0.2.2 protobuf==3.20.3 h5py==2.10.0 biopython==1.73

if [ ! -d /content/alphafold/.git ]; then
    git clone --branch v2.2.4 https://github.com/deepmind/alphafold.git /content/alphafold
fi
if [ ! -x /opt/conda/envs/af2/bin/python ]; then
    conda create -y -n af2 python=3.7
fi
conda activate af2
python -m pip install 'pip<24'
python -m pip install jax==0.3.17 'jaxlib==0.3.15+cuda11.cudnn805' -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
python -m pip install -r /content/alphafold/requirements.txt
python -m pip install --no-deps /content/alphafold
conda install -y -n af2 -c conda-forge python=3.7 cudatoolkit=11.3 cudnn=8.2

apt-get update -qq
apt-get install -y git-lfs unzip
git lfs install
if [ ! -d /content/rgn2_weights/.git ]; then
    GIT_LFS_SKIP_SMUDGE=1 git clone https://huggingface.co/christinafl/rgn2 /content/rgn2_weights
fi
git -C /content/rgn2_weights lfs pull
mkdir -p /content/rgn2/runs/15106000
cp -a /content/rgn2_weights/rgn2_runs/15106000/. /content/rgn2/runs/15106000/

refiner=/content/rgn2/ter2pdb
if [ ! -s "$refiner/ModRefiner-l/emrefinement" ]; then
    curl -fL --retry 3 https://zhanggroup.org/ModRefiner/ModRefiner-l.zip -o /tmp/rgn2-ModRefiner-l.zip
    unzip -o /tmp/rgn2-ModRefiner-l.zip -d "$refiner"
fi
chmod +x "$refiner/ModRefiner-l/emrefinement"

params=/content/alphafold/data/params
mkdir -p "$params"
if [ ! -s "$params/params_model_1_ptm.npz" ]; then
    curl -fL --retry 3 https://storage.googleapis.com/alphafold/alphafold_params_2022-03-02.tar -o /tmp/rgn2-af2.tar.part
    mv /tmp/rgn2-af2.tar.part /tmp/rgn2-af2.tar
    tar -xf /tmp/rgn2-af2.tar -C "$params"
    test -s "$params/params_model_1_ptm.npz"
    rm /tmp/rgn2-af2.tar
fi
printf '\nSetup completed.\n'

## Step 2 — Enter your sequence

In [ ]:
#@title Enter the amino acid sequence to fold

import os
import re
import json
import hashlib
from pathlib import Path

sequence = "QLEDSEVEAVAKGLEEMYANGVTEDNFKNYVKNNFAQQEISSVEEELNVNISDSCVANKIKDEFFAMISISAIVKAAQKKAWKELAVTVLRFAKANGLKTNAIIVAGQLALWAVQCG"  #@param {type:"string"}
jobname = "T1104_rgn"  #@param {type:"string"}

ROOT = Path("/content/rgn2")

if not ROOT.is_dir():
    raise FileNotFoundError(
        "RGN2 is not installed at /content/rgn2. Run the setup first."
    )

os.chdir(ROOT)

# Clean and validate the input.
sequence = "".join(sequence.split()).upper()
jobname = re.sub(r"[^A-Za-z0-9_-]", "", jobname)

if not sequence:
    raise ValueError("Enter a non-empty protein sequence.")

if not jobname:
    jobname = "protein"

allowed_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")
invalid_letters = set(sequence) - allowed_amino_acids

if invalid_letters:
    raise ValueError(
        f"Sequence contains unsupported characters: {sorted(invalid_letters)}"
    )

MAX_SEQUENCE_LENGTH = 1000

if len(sequence) > MAX_SEQUENCE_LENGTH:
    raise ValueError(
        f"Sequence has {len(sequence)} residues. "
        f"This RGN2 configuration supports at most "
        f"{MAX_SEQUENCE_LENGTH} residues."
    )

seq_hash = hashlib.blake2b(
    sequence.encode(), digest_size=3
).hexdigest()

seq_id = f"{jobname}_{seq_hash}"

# Save the authoritative inputs for all later cells.
run_inputs = {
    "sequence": sequence,
    "seq_id": seq_id,
}

with (ROOT / "run.json").open("w") as f:
    json.dump(run_inputs, f, indent=2)

# Use absolute paths throughout.
DATA_DIR = str(ROOT / "aminobert_output")
RUN_DIR = str(ROOT / "runs/15106000")
OUTPUT_DIR = str(ROOT / "output")
REFINE_DIR = str(Path(OUTPUT_DIR) / "refine_model1")

SEQ_PATH = str(Path(DATA_DIR) / f"{seq_id}.fa")
TER_PATH = str(
    Path(RUN_DIR) / "1" / "outputsTesting" / f"{seq_id}.tertiary"
)

print("Sequence ID:", seq_id)
print("Sequence length:", len(sequence))
print("Saved inputs:", ROOT / "run.json")
print("Expected FASTA:", SEQ_PATH)
print("Expected prediction:", TER_PATH)

## Step 3 — Generate AminoBERT embeddings
Previous embeddings are replaced so that only the current sequence is processed.

In [ ]:
%%bash
set -e

source /opt/conda/etc/profile.d/conda.sh
conda activate rgn2
cd /content/rgn2

test -s /content/rgn2/run.json

printf '\nDownloading the AminoBERT checkpoint...\n'

CHECKPOINT_DIR="/content/rgn2/resources/aminobert_checkpoint"
PREFIX="AminoBERT_runs_v2_uniparc_dataset_v2_5-1024_fresh_start_model.ckpt-1100000"
BASE_URL="https://huggingface.co/christinafl/rgn2/resolve/main/aminobert_checkpoint"

mkdir -p "$CHECKPOINT_DIR"

if ! python - <<'PY'
import tensorflow as tf
p = '/content/rgn2/resources/aminobert_checkpoint/AminoBERT_runs_v2_uniparc_dataset_v2_5-1024_fresh_start_model.ckpt-1100000'
reader = tf.train.load_checkpoint(p)
for name in reader.get_variable_to_shape_map():
    reader.get_tensor(name)
print('Existing AminoBERT checkpoint is readable; reusing it.')
PY
then
    for suffix in index data-00000-of-00001 meta; do
        filename="${PREFIX}.${suffix}"
        curl -fL --retry 3 "${BASE_URL}/${filename}" -o "${CHECKPOINT_DIR}/${filename}.part"
        mv "${CHECKPOINT_DIR}/${filename}.part" "${CHECKPOINT_DIR}/${filename}"
    done
fi

printf '\nChecking the checkpoint...\n'

python - <<'PY'
import tensorflow as tf

prefix = (
    "/content/rgn2/resources/aminobert_checkpoint/"
    "AminoBERT_runs_v2_uniparc_dataset_v2_5-1024_"
    "fresh_start_model.ckpt-1100000"
)

variables = tf.train.list_variables(prefix)

if not variables:
    raise RuntimeError("The AminoBERT checkpoint has no readable variables.")

print("Readable checkpoint:", len(variables), "variables")
PY

printf '\nGenerating AminoBERT embeddings...\n'

python - <<'PY'
import os
import sys
import json
import shutil
from pathlib import Path

ROOT = Path("/content/rgn2")
os.chdir(ROOT)

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "aminobert"))

from aminobert.prediction import aminobert_predict_sequence
from data_processing.aminobert_postprocessing import aminobert_postprocess

with (ROOT / "run.json").open() as f:
    run_inputs = json.load(f)

sequence = run_inputs["sequence"]
seq_id = run_inputs["seq_id"]

if not sequence:
    raise ValueError("run.json contains an empty sequence.")

print("Sequence ID:", seq_id)
print("Sequence length:", len(sequence))

# Keep this relative to ROOT for the upstream postprocessing code.
DATA_DIR = "aminobert_output"
DATASET_NAME = "1"
PREPEND_M = True

checkpoint = str(
    ROOT
    / "resources/aminobert_checkpoint"
    / (
        "AminoBERT_runs_v2_uniparc_dataset_v2_5-1024_"
        "fresh_start_model.ckpt-1100000"
    )
)

# Avoid combining previous sequences with the current input.
data_path = ROOT / DATA_DIR

if data_path.exists():
    shutil.rmtree(data_path)

data_path.mkdir(parents=True)

aminobert_predict_sequence(
    seq=sequence,
    header=seq_id,
    prepend_m=PREPEND_M,
    checkpoint=checkpoint,
    data_dir=DATA_DIR,
)

aminobert_postprocess(
    data_dir=DATA_DIR,
    dataset_name=DATASET_NAME,
    prepend_m=PREPEND_M,
)

required_files = [
    data_path / f"{seq_id}.fa",
    data_path / f"{seq_id}.fa.npy",
    data_path / "tfrecord/1",
    ROOT / "data/1",
    ROOT / "data/training/full/1",
]

for path in required_files:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(
            f"Expected non-empty output was not generated: {path}"
        )
    print("Verified:", path)

print("AminoBERT embeddings and RGN2 input files are ready.")
PY

## Step 4 — Apply and verify TensorFlow compatibility fixes

In [ ]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate rgn2
cd /content/rgn2
python - <<'PY'
import inspect
from pathlib import Path
import tensorflow as tf

assert tf.__version__.startswith('1.15'), tf.__version__
tf.compat.v1.disable_v2_behavior()
from tensorflow.contrib.cudnn_rnn.python.layers import cudnn_rnn

def save_with_backup(path, old, new):
    if old != new:
        backup = path.with_name(path.name + '.before_clean_fix')
        if not backup.exists():
            backup.write_text(old)
        compile(new, str(path), 'exec')
        path.write_text(new)

path = Path('/content/rgn2/rgn/protling.py')
text = path.read_text()
anchor = 'import tensorflow as tf\n'
if 'tf.compat.v1.disable_v2_behavior()' not in text:
    assert text.count(anchor) == 1, 'Unexpected protling.py imports'
    patched = text.replace(anchor, anchor + "tf.compat.v1.disable_v2_behavior()\nassert not tf.executing_eagerly()\n", 1)
    save_with_backup(path, text, patched)

path = Path(inspect.getsourcefile(cudnn_rnn))
text = path.read_text()
fixed = "getattr(input_shape[-1], 'value', input_shape[-1])"
patched = text.replace('input_shape[-1].value', fixed)
assert f'if {fixed} is None:' in patched
assert f'self._input_size = {fixed}' in patched
save_with_backup(path, text, patched)
print('Verified RGN2 graph-mode and cuDNN shape fixes.')

checkpoint = tf.train.latest_checkpoint('/content/rgn2/runs/15106000/checkpoints')
assert checkpoint, 'No RGN2 checkpoint discovered'
reader = tf.train.load_checkpoint(checkpoint)
for name in reader.get_variable_to_shape_map():
    reader.get_tensor(name)
for name in ('runs/15106000/configuration', 'data/1', 'data/training/full/1'):
    p = Path(name)
    assert p.is_file() and p.stat().st_size, str(p)
print('RGN2 checkpoint and input files verified:', checkpoint)
PY

## Step 5 — Predict the RGN2 C-alpha structure

In [ ]:
%%bash
set -e

source /opt/conda/etc/profile.d/conda.sh
conda activate rgn2
cd /content/rgn2

# Preserve any earlier result and require a newly generated file.
python - <<'PY'
import json
from pathlib import Path
from datetime import datetime
root = Path('/content/rgn2')
run = json.loads((root / 'run.json').read_text())
p = root / 'runs/15106000/1/outputsTesting' / (run['seq_id'] + '.tertiary')
if p.exists():
    p.rename(p.with_name(p.name + '.previous-' + datetime.now().strftime('%Y%m%d%H%M%S%f')))
PY

status=0

python rgn/protling.py \
    runs/15106000/configuration \
    -p \
    -e weighted_testing \
    -a \
    -g 0 || status=$?

printf '\n=== RGN2 exit code: %s ===\n' "$status"

echo "=== Latest RGN2 log ==="
tail -n 120 /content/rgn2/logs/15106000.log || true

echo "=== Available prediction files ==="
find /content/rgn2/runs/15106000 \
    -type f -name "*.tertiary" -print

if [ "$status" -ne 0 ]; then
    exit "$status"
fi
python - <<'PY'
import json
from pathlib import Path
root = Path('/content/rgn2')
run = json.loads((root / 'run.json').read_text())
p = root / 'runs/15106000/1/outputsTesting' / (run['seq_id'] + '.tertiary')
assert p.is_file() and p.stat().st_size, 'Expected prediction missing: ' + str(p)
print('Prediction completed:', p)
PY

## Step 6 — Verify the refinement GPU environment

In [ ]:
%%bash
set -e

source /opt/conda/etc/profile.d/conda.sh
conda activate af2

export LD_LIBRARY_PATH="/opt/conda/envs/af2/lib${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}"
unset TF_FORCE_UNIFIED_MEMORY
unset XLA_PYTHON_CLIENT_MEM_FRACTION
export XLA_PYTHON_CLIENT_PREALLOCATE=false
export TF_FORCE_GPU_ALLOW_GROWTH=true

nvidia-smi

python -u - <<'PY'
import ctypes
import sys

print("\nPython:", sys.executable)

missing = []

for name in (
    "libcudart.so.11.0",
    "libcublas.so.11",
    "libcusolver.so.11",
    "libcudnn.so.8",
):
    try:
        ctypes.CDLL(name)
        print("Loaded:", name)
    except OSError as error:
        print("FAILED:", name)
        print(error)
        missing.append(name)

if missing:
    raise RuntimeError(
        "CUDA libraries are missing or cannot load. "
        "See the specific errors above."
    )

import jax
import jaxlib
import jax.numpy as jnp
import numpy as np

print("\nJAX:", jax.__version__)
print("JAXLIB:", jaxlib.__version__)

gpu = jax.devices("gpu")[0]
print("Testing GPU:", gpu)

matrix = np.broadcast_to(
    np.diag(np.array([1, 2, 3, 4], dtype=np.float32)),
    (1, 115, 4, 4),
).copy()

matrix = jax.device_put(matrix, gpu)

eigenvalues, eigenvectors = jax.jit(jnp.linalg.eigh)(matrix)
eigenvalues.block_until_ready()
eigenvectors.block_until_ready()

np.testing.assert_allclose(
    np.asarray(eigenvalues)[0, 0],
    [1, 2, 3, 4],
    rtol=1e-5,
)

print("\nPASS: GPU eigenvalue calculation succeeded.")
PY

## Step 7 — Refine and download the final structure
This produces an AlphaFold-refined RGN2 model, not an unrefined RGN2 prediction.

In [ ]:
#@title Refinement and Structure Download

import os
import json
import subprocess
from pathlib import Path
from google.colab import files

recycles = 1  #@param {type:"integer"}

ROOT = Path("/content/rgn2")

with (ROOT / "run.json").open() as f:
    run_inputs = json.load(f)

seq_id = run_inputs["seq_id"]

DATA_DIR = str(ROOT / "aminobert_output")
RUN_DIR = str(ROOT / "runs/15106000")
OUTPUT_DIR = str(ROOT / "output")
REFINE_DIR = str(Path(OUTPUT_DIR) / "refine_model1")

SEQ_PATH = str(Path(DATA_DIR) / f"{seq_id}.fa")
TER_PATH = str(
    Path(RUN_DIR) / "1" / "outputsTesting" / f"{seq_id}.tertiary"
)

RGN2_PYTHON = Path("/opt/conda/envs/rgn2/bin/python")
AF2_PYTHON = Path("/opt/conda/envs/af2/bin/python")

MODREFINER = ROOT / "ter2pdb/ModRefiner-l/emrefinement"
AF2_PARAMS = Path(
    "/content/alphafold/data/params/params_model_1_ptm.npz"
)

CONVERTER_SCRIPT = ROOT / "ter2pdb/ter2pdb.py"
REFINEMENT_SCRIPT = ROOT / "ter2pdb/run_af2rank.py"

print("Sequence ID:", seq_id)
print("FASTA:", SEQ_PATH)
print("RGN2 prediction:", TER_PATH)

# Check prediction inputs.
for label, filename in [
    ("Sequence FASTA", SEQ_PATH),
    ("RGN2 tertiary output", TER_PATH),
]:
    path = Path(filename)
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(
            f"{label} is missing or empty: {path}\n"
            "Complete embeddings and RGN2 prediction before refinement."
        )

# Confirm that the FASTA belongs to the current sequence.
fasta_lines = Path(SEQ_PATH).read_text().splitlines()
fasta_sequence = "".join(
    line.strip()
    for line in fasta_lines
    if not line.startswith(">")
)

if fasta_sequence != run_inputs["sequence"]:
    raise ValueError(
        "The FASTA does not match run.json. "
        "Regenerate embeddings and rerun RGN2."
    )

# Check refinement prerequisites.
for path in [
    RGN2_PYTHON,
    AF2_PYTHON,
    MODREFINER,
    AF2_PARAMS,
    CONVERTER_SCRIPT,
    REFINEMENT_SCRIPT,
]:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(
            f"Required refinement file is missing or empty: {path}\n"
            "Complete the refinement prerequisite setup first."
        )

if not os.access(MODREFINER, os.X_OK):
    raise PermissionError(
        f"ModRefiner is not executable: {MODREFINER}\n"
        f"Run: !chmod +x {MODREFINER}"
    )

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("\nConverting the C-alpha trace to an all-atom structure...")

subprocess.run(
    [
        str(RGN2_PYTHON),
        str(CONVERTER_SCRIPT),
        SEQ_PATH,
        TER_PATH,
        "--output_dir", OUTPUT_DIR,
        "--seq_id", seq_id,
    ],
    cwd=str(ROOT),
    check=True,
    timeout=600,
)

initial_pdb = Path(OUTPUT_DIR) / f"{seq_id}_fulinit.pdb"

if not initial_pdb.is_file() or initial_pdb.stat().st_size == 0:
    raise FileNotFoundError(
        f"ModRefiner did not create the expected structure: {initial_pdb}"
    )

print("Initial structure:", initial_pdb)

out_suffix = "_prediction"

cmd = [
    str(AF2_PYTHON),
    str(REFINEMENT_SCRIPT),
    "refine_model1",
    "--target_list", seq_id,
    "--af2_dir", "/content/alphafold/",
    "--out_suffix", out_suffix,
    "--seq_dir", DATA_DIR,
    "--pdb_dir", OUTPUT_DIR,
    "--output_dir", OUTPUT_DIR,
    "--deterministic",
    "--seq_replacement", "-",
    "--mask_sidechains_add_cb",
    "--model_num", "1",
    "--recycles", str(recycles),
]

env = os.environ.copy()

# Make the af2 environment's shared libraries discoverable.
af2_lib = "/opt/conda/envs/af2/lib"
previous_library_path = env.get("LD_LIBRARY_PATH", "")

env["LD_LIBRARY_PATH"] = (
    af2_lib
    + (":" + previous_library_path if previous_library_path else "")
)

# Remove the previous memory-oversubscription settings.
env.pop("TF_FORCE_UNIFIED_MEMORY", None)
env.pop("XLA_PYTHON_CLIENT_MEM_FRACTION", None)

env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
env["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

# Preserve an earlier PDB; do not accept a stale file as a new result.
from datetime import datetime
final_pdb = Path(REFINE_DIR) / f"{seq_id}{out_suffix}.pdb"
if final_pdb.exists():
    final_pdb.rename(final_pdb.with_name(
        final_pdb.name + '.previous-' + datetime.now().strftime('%Y%m%d%H%M%S%f')
    ))

print("\nRunning AlphaFold-based refinement...")

# Stream output into the notebook as refinement runs.
with subprocess.Popen(
    cmd,
    cwd=str(ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)

    returncode = process.wait()

if returncode != 0:
    raise RuntimeError(
        f"Refinement failed with exit code {returncode}. "
        "Read the output above for the actual error."
    )

final_pdb = Path(REFINE_DIR) / f"{seq_id}{out_suffix}.pdb"

if not final_pdb.is_file() or final_pdb.stat().st_size == 0:
    raise FileNotFoundError(
        f"Refinement did not produce the expected PDB: {final_pdb}"
    )

print("\nRefinement completed:", final_pdb)
files.download(str(final_pdb))

## Step 8 — Download again if needed
This only downloads the existing result; it does not rerun prediction or refinement.

In [ ]:
import json
from pathlib import Path
from google.colab import files

root = Path('/content/rgn2')
run = json.loads((root / 'run.json').read_text())
result = root / 'output/refine_model1' / (run['seq_id'] + '_prediction.pdb')
if not result.is_file() or result.stat().st_size == 0:
    raise FileNotFoundError(f'Complete refinement first: {result}')
files.download(str(result))